In [2]:
# ============================================================
# Gen 6 pre-R1 seam confirmation — full Colab pipeline, one cell
# ============================================================
import os, sys, subprocess, time, json, shutil
import numpy as np

# --- 0. install + GPU check --------------------------------------------
subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                "torch", "transformers", "accelerate", "psutil"], check=False)
import torch
print("torch:", torch.__version__)
assert torch.cuda.is_available(), \
    "no GPU. Runtime → Change runtime type → T4 GPU, then re-run."
print("device:", torch.cuda.get_device_name(0))
print(f"vram: {torch.cuda.mem_get_info()[0]/1e9:.2f} / "
      f"{torch.cuda.mem_get_info()[1]/1e9:.2f} GB free")

# --- 1. file check -----------------------------------------------------
required = ["gen6_full_bench.py", "gen6_v2.py", "stage1_seam_headtohead.py"]
missing = [f for f in required if not os.path.exists(f)]
assert not missing, (f"missing files in /content: {missing}. "
                     f"drag-and-drop them into the Colab file panel, then re-run.")
subprocess.run(["ls", "-la"] + required, check=False)
if "/content" not in sys.path: sys.path.insert(0, "/content")

# --- 2. standalone bench (smoke then full) -----------------------------
print("\n" + "="*60); print("STANDALONE SMOKE  (~3-5 min)"); print("="*60)
assert subprocess.run([sys.executable, "gen6_full_bench.py",
                       "--quick", "--out", "smoke.json"]).returncode == 0, \
       "smoke failed"
print("\n" + "="*60); print("STANDALONE FULL  (~25-40 min)"); print("="*60)
subprocess.run([sys.executable, "gen6_full_bench.py",
                "--out", "results.json"], check=False)

# --- 3. read standalone summary ----------------------------------------
with open("results.json") as f: R = json.load(f)
print("\n" + "="*70); print("STANDALONE SUMMARY"); print("="*70)
for task in ["lorenz", "telemetry", "extreme"]:
    print(f"\n[{task}]")
    for m in ["Gen1", "Gen5", "Gen6"]:
        r = R["tasks"][task][m]
        print(f"  {m:<6} eval={r['eval_mean']:.5f} ± {r['eval_std']:.5f}   "
              f"diverged={r['n_diverged']}/{len(r['seeds'])}")
print("\n[continual]")
for m in ["Gen1", "Gen5", "Gen6"]:
    rows = R["tasks"]["continual"][m]["seeds"]
    traj = np.array([r["task0_over_time"] for r in rows], dtype=float)
    print(f"  {m:<6} " + " → ".join(f"{x:.4f}" for x in np.nanmean(traj, axis=0)))
print("\nGen 6 certificates (last seed, extreme task):")
print(json.dumps(R["tasks"]["extreme"]["Gen6"]["seeds"][-1]["audit"], indent=2))

# --- 4. R1 weights -----------------------------------------------------
if not os.path.isdir("./r1-14b"):
    print("\n" + "="*60); print("DOWNLOADING R1 (~28 GB, ~10-15 min)"); print("="*60)
    subprocess.run(["huggingface-cli", "download",
                    "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B",
                    "--local-dir", "./r1-14b"], check=False)
else:
    print("\n✓ R1 already at ./r1-14b")
subprocess.run(["du", "-sh", "./r1-14b"], check=False)

# --- 5. seam ablation (smoke then full) --------------------------------
print("\n" + "="*60); print("SEAM SMOKE  (~3 min)"); print("="*60)
assert subprocess.run([sys.executable, "stage1_seam_headtohead.py",
                       "--model_dir", "./r1-14b", "--layer", "20",
                       "--quick", "--out", "stage1_smoke.json"]).returncode == 0, \
       "seam smoke failed"
print("\n" + "="*60); print("SEAM FULL  (~10 min)"); print("="*60)
subprocess.run([sys.executable, "stage1_seam_headtohead.py",
                "--model_dir", "./r1-14b", "--layer", "20",
                "--out", "stage1_results.json"], check=False)

# --- 6. read seam summary ----------------------------------------------
with open("stage1_results.json") as f: S = json.load(f)
print("\n" + "="*70); print("SEAM SUMMARY"); print("="*70)
for c in ["random", "Gen1", "Gen5", "Gen6_ChartB"]:
    kls, flips, drifts = [], [], []
    for pr in S["by_probe"]:
        cmp = pr["comparisons_vs_zeroed"].get(c)
        if cmp is None: continue
        kls.append(cmp["kl_vs_zeroed"])
        flips.append(cmp["argmax_flip_vs_zeroed"])
        if cmp["downstream_drift_L2"]:
            drifts.append(list(cmp["downstream_drift_L2"].values())[-1])
    print(f"  {c:<14}  KL={np.mean(kls):.4f}   flip={np.mean(flips):.2f}   "
          f"drift_downstream={np.mean(drifts):.4f}")
print("\n  random KL ~ 0     -> hook misconfigured")
print("  arch KL ~ 0       -> READ PROBLEM (host ignores learned content)")
print("  Gen6 drift < Gen1 -> contraction certificate visible at the seam")

# --- 7. save to Drive --------------------------------------------------
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ts = time.strftime("%Y%m%d_%H%M%S")
    dest = f"/content/drive/MyDrive/recursion_runs/{ts}"
    os.makedirs(dest, exist_ok=True)
    for f in ["results.json", "stage1_results.json",
              "smoke.json", "stage1_smoke.json"]:
        if os.path.exists(f):
            shutil.copy(f, dest); print(f"saved → {dest}/{f}")
except Exception as e:
    print("drive save skipped:", e)

print("\n✓ DONE")

torch: 2.11.0+cu128
device: Tesla T4
vram: 15.53 / 15.64 GB free


AssertionError: missing files in /content: ['gen6_full_bench.py', 'gen6_v2.py', 'stage1_seam_headtohead.py']. drag-and-drop them into the Colab file panel, then re-run.